# Gemma 4 27B-A4B Tool-Calling Fine-Tune (Kaggle T4)

Fine-tune Google's **Gemma 4 27B-A4B** (Mixture-of-Experts, ~4B active params) for tool-calling
using Unsloth + QLoRA on Kaggle's free T4 GPU.

> **Why 27B-A4B instead of 31B?** The 31B dense model needs ~20GB+ VRAM for 4-bit QLoRA.
> Kaggle's free T4 has 16GB. The 27B-A4B MoE model has only ~4B active params per forward pass,
> making it fast and memory-efficient while retaining strong quality.

**Setup:** Settings → Accelerator → GPU T4 × 2 → Internet ON → Persistence: Files only

In [ ]:
# === Install Unsloth (Kaggle CUDA environment) ===
# Kaggle already has PyTorch + CUDA. Just add Unsloth.
%%capture
!pip install --no-deps unsloth unsloth-zoo
!pip install --no-deps "git+https://github.com/unslothai/unsloth-zoo.git"
!pip install "unsloth[kaggle] @ git+https://github.com/unslothai/unsloth"
!pip install --upgrade git+https://github.com/huggingface/transformers.git

In [ ]:
import torch, unsloth, transformers
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} ({props.total_mem // 1024**3}GB)")
print(f"Unsloth: {getattr(unsloth, '__version__', 'OK')}")
print(f"Transformers: {transformers.__version__}")

In [ ]:
import os

# === HuggingFace Configuration ===
# Option 1: Kaggle Secrets (recommended) - Add HF_TOKEN in Settings → Secrets
# Option 2: Set directly (not recommended for shared notebooks)
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ HF_TOKEN loaded from Kaggle Secrets")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if HF_TOKEN:
        print("✅ HF_TOKEN loaded from environment")
    else:
        print("⚠️  No HF token. Hub pushing disabled.")

HF_REPO_ID = "mtita/gemma4-27b-a4b-tool-calling-lora"
PUSH_TO_HUB = bool(HF_TOKEN) and HF_TOKEN != "YOUR_HF_TOKEN"

# === Training Output ===
# /kaggle/working persists between sessions with "Files only" persistence
TRAINING_OUTPUT_DIR = "/kaggle/working/gemma4_runs/gemma4_toolcalling_sft"
os.makedirs(TRAINING_OUTPUT_DIR, exist_ok=True)

print(f"HF repo: {HF_REPO_ID}")
print(f"Push enabled: {PUSH_TO_HUB}")
print(f"Training outputs: {TRAINING_OUTPUT_DIR}")

### Load Model

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-27B-A4B-it",
    dtype=None,
    max_seq_length=4096,
    load_in_4bit=True,
    full_finetuning=False,
    token=HF_TOKEN or None,
)

### Add LoRA Adapters

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_gradient_checkpointing = "unsloth",
)

### Data Prep

We use the `gemma-4` chat template. Gemma 4 renders conversations like:
```
<bos><|turn>user
Hello<turn|>
<|turn>model
Hey there!<turn|>
```

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

In [ ]:
from datasets import load_dataset

# Use full dataset. Reduce split for faster testing: "train[:3000]"
DATASET_SPLIT = "train"
dataset = load_dataset(
    "mtita/gemma4-tool-calling-training",
    data_files="combined_training.jsonl",
    split=DATASET_SPLIT,
)
print(f"Loaded {len(dataset)} training examples from {DATASET_SPLIT}")
print(dataset[0]["messages"][:2])

Map roles for Gemma 4 (system → user prefix, tool → user prefix) and apply chat template.

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        formatted = []
        for msg in messages:
            role = msg["role"]
            content = msg.get("content", "") or ""
            if role == "system":
                sys_msg = {"role": "user", "content": f"[System Instructions]\n{content}"}
                ack_msg = {"role": "assistant", "content": "Understood."}
                if formatted and formatted[-1]["role"] == "user":
                    formatted[-1]["content"] += f"\n\n{sys_msg['content']}"
                    formatted.append(ack_msg)
                else:
                    formatted.append(sys_msg)
                    formatted.append(ack_msg)
            elif role == "user":
                if formatted and formatted[-1]["role"] == "user":
                    formatted[-1]["content"] += f"\n\n{content}"
                else:
                    formatted.append({"role": "user", "content": content})
            elif role == "assistant":
                if formatted and formatted[-1]["role"] == "assistant":
                    formatted[-1]["content"] += f"\n\n{content}"
                else:
                    formatted.append({"role": "assistant", "content": content})
            elif role == "tool":
                tool_name = msg.get("name", "tool")
                tool_text = f"[Tool Result: {tool_name}]\n{content}"
                if formatted and formatted[-1]["role"] == "user":
                    formatted[-1]["content"] += f"\n\n{tool_text}"
                else:
                    formatted.append({"role": "user", "content": tool_text})
        if not formatted or formatted[0]["role"] != "user":
            texts.append("")
            continue
        if formatted[-1]["role"] != "assistant":
            texts.append("")
            continue
        try:
            text = tokenizer.apply_chat_template(
                formatted, tokenize=False, add_generation_prompt=False
            ).removeprefix('<bos>')
            texts.append(text)
        except Exception:
            texts.append("")
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
dataset = dataset.filter(lambda x: len(x["text"]) > 50)
print(f"Formatted: {len(dataset)} examples")
print(dataset[0]["text"][:500])

### Training Setup

Training runs for **2 full epochs**. Checkpoints save every 50 steps.
If the kernel dies, re-run cells from the top — checkpoint resume picks up automatically.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

LAST_CHECKPOINT = get_last_checkpoint(TRAINING_OUTPUT_DIR)
print("Latest checkpoint:", LAST_CHECKPOINT or "none")

Only train on assistant outputs, ignore user inputs:

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        warmup_steps=5,
        learning_rate=1e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
        output_dir=TRAINING_OUTPUT_DIR,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=5,
        hub_model_id=HF_REPO_ID,
        push_to_hub=PUSH_TO_HUB,
        hub_strategy="checkpoint",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

### Verify masking before training

In [ ]:
# Full conversation visible
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

In [ ]:
# Only assistant responses visible (rest is spaces = masked)
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]])

### Start or resume training

If the kernel dies, re-run all cells above, then run this cell again.
It will automatically resume from the latest checkpoint.

In [ ]:
train_result = trainer.train(resume_from_checkpoint=LAST_CHECKPOINT)
train_result

### Push final adapter to Hugging Face

In [ ]:
if PUSH_TO_HUB:
    trainer.push_to_hub()
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print("Pushed final training artifacts to", HF_REPO_ID)
else:
    print("⚠️ PUSH_TO_HUB is disabled. Set HF_TOKEN in Kaggle Secrets to enable.")

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_mem / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Max memory: {max_memory} GB")
print(f"Peak reserved: {start_gpu_memory} GB")
print(f"Used: {round(start_gpu_memory / max_memory * 100, 1)}%")

### Test Inference

In [ ]:
messages = [{"role": "user",
"content": [{"type": "text", "text": "You have these tools:\n"
"- read(path): Read a file\n"
"- edit(path, old, new): Edit a file\n\n"
"Please read the file config.yaml and fix the port to 8080."}]}]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    input_ids=inputs, streamer=text_streamer,
    max_new_tokens=512, temperature=0.6, top_p=0.9,
    use_cache=True,
)

### Save LoRA adapters

In [ ]:
model.save_pretrained("/kaggle/working/gemma_4_lora")
tokenizer.save_pretrained("/kaggle/working/gemma_4_lora")

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print(f"Pushed to {HF_REPO_ID}")

### Export to GGUF

Supported: q4_k_m (recommended), q5_k_m, q8_0, f16, bf16 and more.

In [ ]:
model.save_pretrained_gguf(
    "/kaggle/working/gemma_4_finetune",
    tokenizer,
    quantization_method="q4_k_m",
)

In [ ]:
if PUSH_TO_HUB:
    model.push_to_hub_gguf(
        "mtita/gemma4-27b-a4b-tool-calling-q4_k_m-GGUF",
        tokenizer,
        quantization_method="q4_k_m",
        token=HF_TOKEN,
    )
    print("GGUF pushed to Hub")
else:
    print("⚠️ PUSH_TO_HUB disabled — GGUF saved locally only")